论文 **《DiffusionNFT: Online Diffusion Reinforcement with Forward Process》** 是对齐技术在扩散模型（Diffusion/Flow Matching）领域的一大突破。

它敏锐地发现了现有扩散模型在线强化学习（如 FlowGRPO、DanceGRPO）的底层痛点，并巧妙地将 **负向偏好微调（NFT，Negative Preference Tuning）** 的哲学引入了**前向扩散过程（Forward Process）**，用纯粹的有监督流匹配（Flow Matching）统一了正负反馈。

以下是该论文的核心要点与数学公式的深度详解：

---

## 一、 核心动机：现有扩散 RL（如 FlowGRPO）的四大痛点

传统的扩散模型在线强化学习（如 FlowGRPO）本质上是在反向采样过程（Reverse Process）上做文章。它们为了计算策略梯度（Policy Gradient），必须把确定性的 ODE 采样器强行注入噪声变成随机 SDE，从而带来了严重的副作用：

* **采样器严重受限（Solver Restriction）**：由于需要精确估计路径的对数似然（Log-likelihood），训练时**只能使用一阶简陋的 SDE 采样器**（如 Euler-Maruyama），根本无法享受 DDIM、DPM-Solver++ 等现代高级高阶黑盒 ODE 采样器的提速红利。
* **内存与计算暴死（Memory Inefficiency）**：由于损失函数依赖于整条反向轨迹，算法在数据收集阶段必须**把整条采样路径上所有时间步的中间状态 $x_t$ 全部存在显存里**，显存开销呈线性爆炸。
* **前向-反向不一致（Forward-Reverse Inconsistency）**：在反向离散路径上强行拉扯策略梯度，极易破坏扩散模型底层的 Fokker-Planck 方程一致性，导致模型在多轮迭代后退化崩溃。

### 💡 DiffusionNFT 的解法

跳出反向过程的泥潭，将在线 RL 彻底搬到**前向加噪过程（Forward Process）**中。
在数据收集期，你可以用**任意黑盒高阶 ODE 采样器**一口气生成最终图像。在训练期，**只需要拿到最终的干净图像 $x_0$ 和对应的 Reward**，然后直接套用前向流匹配目标进行训练。这使得采样与训练完全解耦，显存开销骤降。

---

## 二、 核心数学公式推导

DiffusionNFT 的核心精髓在于：**利用群组奖励（Group Reward）将样本划分为隐式的正负两极，并通过隐式参数化（Implicit Parameterization）让单个网络同时学会"向专家看齐"与"远离危险边界"。**

### 2.1 群组奖励归一化与软标签映射

对于同一个 Prompt $c$，模型同时采样生成 $K$ 个样本。首先计算其原始 Reward $\{r_{\text{raw}}\}_{1:K}$，并在组内进行均值归一化（延续了 GRPO 的精髓）：


$$r_{\text{norm}} = r_{\text{raw}} - \text{mean}(\{r_{\text{raw}}\}_{1:K})$$

为了将其转化为 NFT 可以处理的二元偏好概率，论文引入了一个带截断的线性映射，将奖励软化为 $r \in [0, 1]$ 的伪标签：


$$r = 0.5 + 0.5 \cdot \text{clip}\left(\frac{r_{\text{norm}}}{Z_c}, -1, 1\right)$$

* 当 $r \to 1$ 时，代表该样本是组内的**高回报正样本（专家行为）**；
* 当 $r \to 0$ 时，代表该样本是组内的**低回报负样本（危险/低劣行为）**。

### 2.2 隐式速度导向（Implicit Velocity Steering）

在语言模型的 NFT 中，我们通过拉扯新旧模型的 Token 概率来建立对立。而在连续空间的流匹配（Flow Matching）中，模型预测的是轨迹的切线速度 $v_\theta(x_t, c, t)$。

论文并没有训练两个独立的网络，而是通过**隐式参数化**，用当前可导网络 $v_\theta$ 和行为策略网络（收集数据的旧网络）$v_{\text{old}}$ 线性组合出虚拟的"正极速度 $v^+$"和"负极速度 $v^-$"：


$$v^+ = (1 - \beta) v_{\text{old}}(x_t, c, t) + \beta v_\theta(x_t, c, t)$$

$$v^- = (1 + \beta) v_{\text{old}}(x_t, c, t) - \beta v_\theta(x_t, c, t)$$

其中 $\beta > 0$ 是控制对齐拉扯强度的超参数。

> **这里的代数结构非常优雅：**
> * 在正极 $v^+$ 中，$v_\theta$ 的系数为正，意味着目标是让当前模型去顺应、放大超过 $v_{\text{old}}$ 的专家梯度。
> * 在负极 $v^-$ 中，$v_\theta$ 的系数变为了 $-\beta$，这意味着当前模型在面对差评样本时，会被强行推向 $v_{\text{old}}$ 的反方向，实现**镜像式的负向惩罚（Negative Preference Alignment）**。
> 
> 

### 2.3 联合前向流匹配损失函数（The Unified Loss）

将上述隐式对立速度代入到标准的前向流匹配（Flow Matching）监督损失中，利用软标签 $r$ 和 $1-r$ 进行动态加权平衡：


$$\mathcal{L}_{\text{DiffusionNFT}}(\theta) = \mathbb{E}_{x_0 \sim \pi_{\text{old}}, \epsilon \sim \mathcal{N}(0, \mathbf{I}), c, t} \left[ r \|v^+ - v\|^2 + (1 - r) \|v^- - v\|^2 \right]$$

其中 $v = \epsilon - x_0$ 是 Rectified Flow 的标准前向真实速度目标（如果是标准 SDE 扩散，则对应为噪声或得分目标）。

通过对该损失函数求导可以发现：**它在微积分空间里的梯度，完美复刻了在线强化学习（RL）通过优势函数（Advantage）对速度场进行推拉的动力学行为。** 从而实现了"用监督学习的代码和稳定性，白嫖在线 RL 性能"的目的。

---

## 三、 为什么 DiffusionNFT 能带来高达 25 倍的效率提升？

1. **全面解放采样器（Solver-Agnostic）**：在数据收集阶段，它可以使用 DPM-Solver++ 等高级器，仅需 20~40 步即可吐出高质量在线样本；而 FlowGRPO 必须使用 100~200 步的随机 Euler-Maruyama 采样。仅在采样开销上，DiffusionNFT 就实现了数倍的绝对剪枝。
2. **免除无条件分支（CFG-Free Alignment）**：传统的扩散对齐非常依赖 Classifier-Free Guidance（CFG），在推理时需要同时前向计算条件和无条件两个分支。而 DiffusionNFT 的隐式流匹配推拉，**直接把正负对抗的指导信号熔断进了单网络参数 $\theta$ 内部**。实验表明，DiffusionNFT 在完全关闭 CFG 的情况下，对齐指标依然显著超越了开启 CFG 的传统基线。
3. **工程鲁棒性极高**：因为 Loss 的核心形式是 $\| \cdot \|^2$ 的均方误差回归，它完美规避了传统 DPO/RLHF 在离线/弱在线场景下由于指数项 $\exp(\cdot)$ 带来的梯度过载和奖励黑客（Reward Hacking）风险。在 GenEval 评测中，DiffusionNFT 仅用 1k 步就将得分从 0.24 飙升至 0.98，而 FlowGRPO 花了 5k 步也仅能达到 0.95。

你觉得这种将在线 RL 转化为前向流匹配回归的思路，如果迁移到更复杂的连续控制（例如自动驾驶的碰撞规避与安全轨迹生成）中，它的隐式速度导向（$v^+$ 和 $v^-$）应该如何针对多目标奖励（如速度、舒适度、碰撞率）进行更精细的解耦和权衡呢？

## Sergey Levine 提出的 **"控制即推断"（Control as Inference）** 框架的数学推导与物理意义

该框架的核心思想是：**将传统强化学习中"最大化连续奖励分数"的决策问题，完美转化为概率图模型中的"条件概率推断"问题。** 从而为后续在连续流匹配（Flow Matching）空间中直接进行有监督形式的拉扯对抗奠定了数学基石。

---

## 四、 核心概念与符号定义

为了进行严密的数学推导，我们首先定义概率空间中的核心变量：

* $\mathbf{c}$：给定的上下文/条件（Context / Prompt）。
* $\mathbf{x}_0$：模型生成的最终样本（图像、文本或自动驾驶轨迹）。
* $\pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c})$：**行为策略（Behavior Policy）**，即当前用于在线采样生成数据的老模型。
* $\mathbf{o} \in \{0, 1\}$：引入的虚拟二元随机变量，称为 **最优性（Optimality）**。
* $\mathbf{o} = 1$：代表样本是"完美的、绝对正确的"。
* $\mathbf{o} = 0$：代表样本是"不完美的、低劣的"。


* $r(\mathbf{x}_0, \mathbf{c}) \in [0, 1]$：**软化奖励函数**。在控制即推断框架下，连续的奖励分数被赋予了明确的概率学定义：

$$r(\mathbf{x}_0, \mathbf{c}) := p(\mathbf{o} = 1 | \mathbf{x}_0, \mathbf{c})$$



---

## 五、 核心数学推导过程

基于上述定义，我们通过**概率乘法公式**、**全概率公式**以及**贝叶斯定理**，推导在线数据在"抛硬币"软划分下所收敛到的理想条件概率密度。

### 5.1 建立生成与最优的联合概率（Joint Probability）

当老模型 $\pi^{\text{old}}$ 吐出一个样本 $\mathbf{x}_0$ 时，该样本本身被创造出来的概率密度为 $p(\mathbf{x}_0 | \mathbf{c}) = \pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c})$。

根据概率论的条件乘法公式（即 $P(AB) = P(B|A)P(A)$），**"模型既生成了 $\mathbf{x}_0$，且该样本同时被判定为完美（进入正样本宇宙）"** 的联合概率密度为：


$$p(\mathbf{x}_0, \mathbf{o}=1 | \mathbf{c}) = p(\mathbf{o}=1 | \mathbf{x}_0, \mathbf{c}) \cdot p(\mathbf{x}_0 | \mathbf{c})$$

将定义代入上式，可得：


$$p(\mathbf{x}_0, \mathbf{o}=1 | \mathbf{c}) = r(\mathbf{x}_0, \mathbf{c}) \cdot \pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c})$$

### 5.2 计算全场平均胜率（Marginal Probability）

为了将上述联合概率标准化为合格的概率密度函数，我们需要求出其归一化分母。在大数定律下，老模型 $\pi^{\text{old}}$ 在当前上下文 $\mathbf{c}$ 下生成的所有样本中，**能被判定为完美的总概率（边缘概率）**，通过全概率公式对整个样本空间积分得到：


$$p_{\text{old}}(\mathbf{o}=1 | \mathbf{c}) = \int p(\mathbf{x}_0, \mathbf{o}=1 | \mathbf{c}) d\mathbf{x}_0 = \int r(\mathbf{x}_0, \mathbf{c}) \cdot \pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c}) d\mathbf{x}_0$$

这一步的物理意义即为老模型当前策略的"全场平均得分（胜率）"。

### 5.3 导出后验条件概率（Posterior Distribution）

现在，我们应用**贝叶斯定理**。已知一个样本已经成功通过了最优性筛选（即已知 $\mathbf{o}=1$），那么**该样本长成 $\mathbf{x}_0$ 模样的后验条件概率密度** $\pi^+(\mathbf{x}_0 | \mathbf{c})$ 展开为：


$$\pi^+(\mathbf{x}_0 | \mathbf{c}) := p(\mathbf{x}_0 | \mathbf{o}=1, \mathbf{c}) = \frac{p(\mathbf{x}_0, \mathbf{o}=1 | \mathbf{c})}{p_{\text{old}}(\mathbf{o}=1 | \mathbf{c})}$$

将步骤一与步骤二的结果代入，即可完美导出论文中的**理想专家正策略公式**：


$$\pi^+(\mathbf{x}_0 | \mathbf{c}) = \frac{r(\mathbf{x}_0, \mathbf{c})}{p_{\text{old}}(\mathbf{o}=1 | \mathbf{c})} \pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c})$$

---

### 镜像推导：负样本分布 $\pi^-$

同理，对于被判定为不完美（$\mathbf{o}=0$）的负样本宇宙，其硬币反面朝上的概率为 $p(\mathbf{o}=0 | \mathbf{x}_0, \mathbf{c}) = 1 - r(\mathbf{x}_0, \mathbf{c})$。
全场的平均失败率为 $1 - p_{\text{old}}(\mathbf{o}=1 | \mathbf{c})$。

通过完全相同的贝叶斯后验推导：


$$\pi^-(\mathbf{x}_0 | \mathbf{c}) := p(\mathbf{x}_0 | \mathbf{o}=0, \mathbf{c}) = \frac{p(\mathbf{x}_0, \mathbf{o}=0 | \mathbf{c})}{p_{\text{old}}(\mathbf{o}=0 | \mathbf{c})}$$

$$\pi^-(\mathbf{x}_0 | \mathbf{c}) = \frac{1 - r(\mathbf{x}_0, \mathbf{c})}{1 - p_{\text{old}}(\mathbf{o}=1 | \mathbf{c})} \pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c})$$

至此，理想的正负基准策略在数学上被完全确立。

---

## 六、 物理意义："虚拟平行宇宙"的软划分机制

在工程实践中，上述数学推导带来了一种极其优雅的**数据流动机制**：

| 特性 | 传统拒绝微调 (RFT) | 扩散 NFT (基于控制即推断) |
| --- | --- | --- |
| **数据利用** | 硬截断（只保留高分样本，扔掉低分样本） | 软划分（全量保留，按概率分流） |
| **样本命运** | 一个样本要么是正面教材，要么被销毁 | 一个样本根据其奖励 $r$，同时具备正负两种"人格" |
| **收敛目标** | 强行逼近局部高分区域的单峰分布 | 在全图空间上拉扯出 $\pi^+ \succ \pi^{\text{old}} \succ \pi^-$ 的动态场 |

### 机制工作流：

1. **在线生成**：模型依据 $\pi^{\text{old}}$ 生成一个样本，并获得一个连续分值 $r=0.8$。
2. **期望收敛**：不需要真实建立两个物理集装箱。在微积分的损失函数中，该样本在期望上有 **80% 的拉力**将当前模型导向专家分布 $\pi^+$，同时有 **20% 的推力**促使模型远离边界负分布 $\pi^-$。
3. **密度扭曲**：无数样本累积后，原本由老模型均匀撒下的样本点，在特征空间中被 $r$ 和 $1-r$ 动态重构，使高回报区域的样本密度天然收敛为 $\pi^+$ 曲线，低回报区域的样本密度天然收敛为 $\pi^-$ 曲线。

---

## 七、 总结：从决策到推断的范式转变

Sergey Levine 框架的最终胜利在于：它向我们证明了，**我们不需要在代码里写任何复杂的在线策略梯度算法（如 PPO）来动态调整策略。** 只要我们利用贝叶斯公式确立了 $\pi^+$ 和 $\pi^-$ 的存在，扩散模型或流匹配网络就可以直接利用其强大的**条件概率拟合能力**，通过有监督的均方误差（MSE）损失函数，同时去拟合向 $\pi^+$ 靠近的速度场 $v^+$ 与远离 $\pi^-$ 的速度场 $v^-$。这用纯粹的监督学习代码，完美"白嫖"了在线强化学习的全部红利。

现在我们把目光收回到**第一章核心段落（Section 3.1 理论桥梁段）**。这一段在整篇论文中扮演着"承上启下"的关键角色。它说的是：**我们虽然用 Levine 的框架得到了正负宇宙，但传统的"只吸取正面教训"的做法（RFT）是有缺陷的，因此我们要利用扩散模型特有的"速度场"，引入负面反馈来强行推导一个改进方向 $\Delta$。**

我们把它拆成三个层级来彻底吃透：

---

## 八、 策略提升的绝对主线：什么叫 $\pi^* \succ \pi^{\text{old}}$？

在强化学习里，我们每一步迭代的终极目标就四个字：**策略提升（Policy Improvement）**。
如何衡量一个新策略 $\pi^*$ 比老策略 $\pi^{\text{old}}$ 好？就看它生成的样本在当前环境（$c$）下的**数学期望得分**是不是更高：


$$\mathbb{E}_{\pi^*(\cdot|\mathbf{c})}r(\mathbf{x}_0, \mathbf{c}) > \mathbb{E}_{\pi^{\text{old}}(\cdot|\mathbf{c})}r(\mathbf{x}_0, \mathbf{c})$$

作者用了一个非常形象的符号来表达这种"血统压制"：**$\pi^* \succ \pi^{\text{old}}$**（即 $\pi^*$ 优于 $\pi^{\text{old}}$）。

---

## 九、 传统做法的瓶颈：拒绝微调（RFT）的单向车道

根据前文用贝叶斯定理推导出的正负宇宙分布，作者指出，数学上可以非常轻松地证明一个恒等的三阶压制链条：


$$\pi^+ \succ \pi^{\text{old}} \succ \pi^-$$

* **$\pi^+$（完美的后验分布）** 必然优于 **$\pi^{\text{old}}$（当前的老模型）**；
* **$\pi^{\text{old}}$（当前的老模型）** 又必然优于 **$\pi^-$（崩坏的垃圾分布）**。

既然 $\pi^+$ 是完美的化身，那最简单粗暴的改进方法就是直接让 $\pi^* = \pi^+$。

> **这就是前人做过的 Rejection Fine-Tuning (RFT，拒绝微调)**：
> 在线生成一堆数据，用硬门槛或硬抽样把掉进正样本集 $\mathcal{D}^+$ 的数据留下来，然后让模型在 $\mathcal{D}^+$ 上老老实实做有监督的扩散训练（Diffusion Training）。

### ❌ RFT 的痛点：

这种做法虽然简单，但是它**完全浪费了 $\mathcal{D}^-$（负样本集）里的信息**。在数据驱动的 AI 中，"知道怎么做会死"和"知道怎么做能活"同等重要。不利用负面反馈，模型在遇到未知的危险边界时，依然容易由于缺乏警惕而踩坑。

---

## 十、 本文的核武器：强化指导（Reinforcement Guidance）与速度场拉扯

为了同时把 $\mathcal{D}^+$ 和 $\mathcal{D}^-$ 的全量信息压榨干净，作者在扩散模型/流匹配（Diffusion / Flow Matching）的语境下提出了一个极其惊艳的公式 (3)：

$$v^*(\mathbf{x}_t, \mathbf{c}, t) := v^{\text{old}}(\mathbf{x}_t, \mathbf{c}, t) + \frac{1}{\beta}\Delta(\mathbf{x}_t, \mathbf{c}, t)$$

这个公式的工程物理图像非常优美：

* **$v^{\text{old}}(\mathbf{x}_t, \mathbf{c}, t)$**：这是老模型在时步 $t$、状态 $\mathbf{x}_t$ 下原本打算走的一步**基础速度（原始意志）**。
* **$\Delta(\mathbf{x}_t, \mathbf{c}, t)$**：作者称之为 **强化指导（Reinforcement Guidance）**。它是一个高维空间里的**方向向量**，它的任务是告诉模型："往这边走能靠向 $\pi^+$，背离这边走能远离 $\pi^-$"。
* **$\frac{1}{\beta}$**：**指导强度（Guidance Strength）**。用来控制这个外加的拉扯力量到底有多大。

### 💡 为什么说它像 Classifier-Free Guidance (CFG)？

在图像生成里，CFG 的公式是 $v = v_{\text{uncond}} + s \cdot (v_{\text{cond}} - v_{\text{uncond}})$，通过两股速度的差值来强行放大文本控制力。
作者在这里完美复刻了这个魔术：**我们不需要像传统 RL 那样去小心翼翼地修改策略的概率分布，我们直接在扩散模型的"速度场（Velocity Field）"上做加法！** 只要基础速度加上了这个修正项 $\Delta$，最终积分生成的样本就一定会自然而然地朝着高回报的方向演化。

---

## 十一、 留下悬念：接下来的两座大山

这一段的最后，作者直接抛出了接下来 Section 3.2 要解决的核心硬骨头：

1. **这个 $\Delta$ 到底长什么样？**（怎么用数学形式把 $\mathcal{D}^+$ 的吸引力和 $\mathcal{D}^-$ 的排斥力完美融合进一个 $\Delta$ 里？）
2. **在工程上，怎么通过 PyTorch 的有监督 Loss 直接把网络参数 $\theta$ 逼近到这个理想的 $v^*$ 速度场上？**

这就是你发给我的这段话的完整全貌：**它指出了传统只看正面教材的局限（RFT），并正式提出了在扩散模型速度场上通过外加 $\Delta$ 向量来进行"强化拉扯"的全新范式。**

这一部分（Section 3.2 **NEGATIVE-AWARE DIFFUSION REINFORCEMENT WITH FORWARD PROCESS**）是整篇论文最具原创性和数学美感的**核心定理段**。

前文提到，为了同时压榨正负样本的信息，我们需要在速度场上加一个修正项 $\Delta$。而这一节的任务，就是彻底解构 **$\Delta$ 在连续时间、连续扩散/流匹配空间里的几何长相，并证明为什么它可以兼顾"正向吸引"与"负向排斥"**。

我们将这部分的硬核数学与几何图像做最通俗的拆解：

---

## 十二、 核心定理：Theorem 3.1（改进方向的几何对齐）

作者丢出了一个惊人的数学结论：在扩散模型的任意一个中间时间步 $t$ 和状态 $\mathbf{x}_t$ 下，**向好样本靠近的拉力，和远离坏样本的推力，在方向上是完全共线的（Proportional）**！

公式 (4) 表达了这种完美的等价性：


$$\Delta := [1 - \alpha(\mathbf{x}_t)] \big[ v^{\text{old}}(\mathbf{x}_t, \mathbf{c}, t) - v^-(\mathbf{x}_t, \mathbf{c}, t) \big]$$

$$\quad = \alpha(\mathbf{x}_t) \big[ v^+(\mathbf{x}_t, \mathbf{c}, t) - v^{\text{old}}(\mathbf{x}_t, \mathbf{c}, t) \big]$$

### 12.1 物理几何层面的拉扯（对照你的 Figure 3 看）：

我们把高维空间想象成一个平面，此时有三个速度向量：

* $v^{\text{old}}$：老模型原本要走的方向（紫色箭头）。
* $v^+$：通往完美宇宙（$\mathcal{D}^+$）的理想速度（红色虚线箭头）。
* $v^-$：通往垃圾堆（$\mathcal{D}^-$）的错误速度（蓝色虚线箭头）。

公式 (4) 告诉我们：

* **右半部分 $\big[ v^+ - v^{\text{old}} \big]$**：代表**正向吸引力**。即"从我当前的位置，修正到完美速度还差多少"。
* **左半部分 $\big[ v^{\text{old}} - v^- \big]$**：代表**负向排斥力**。即"我要怎么走，才能离那个垃圾速度越远越好"。

**这两股力量，乘以各自的系数后，在空间中合成了同一个宏观方向向量 $\Delta$（图中的灰色虚线大箭头 Guidance $\Delta$）**！

---

## 十三、 关键系数 $\alpha(\mathbf{x}_t)$ 的深层含义

公式中引入了一个动态标量系数 $0 \le \alpha(\mathbf{x}_t) \le 1$：


$$\alpha(\mathbf{x}_t) := \frac{\pi_t^+(\mathbf{x}_t | \mathbf{c})}{\pi_t^{\text{old}}(\mathbf{x}_t | \mathbf{c})} \mathbb{E}_{\pi^{\text{old}}(\mathbf{x}_0 | \mathbf{c})} r(\mathbf{x}_0, \mathbf{c})$$

把我们在第一步推导出的 Levine 框架下的 $\pi^+$ 表达式代入进去，你会发现这个 $\alpha(\mathbf{x}_t)$ 在中间时步的本质是：**当前状态 $\mathbf{x}_t$ 最终能够演化成一个高分完美样本的"概率期望"或"置信度"**。

### 💡 随着 $\alpha(\mathbf{x}_t)$ 强弱变化的动态流转（软划分的精髓）：

* **当 $\alpha(\mathbf{x}_t) \to 1$ 时（模型处于安全、高分的黄金区域）**：
公式 (4) 的左半部分 $[1-\alpha]$ 趋近于 0。此时排斥力不怎么起作用，主要由右半部分的**正向吸引力 $\big[ v^+ - v^{\text{old}} \big]$** 来主导。模型的主要任务是"精益求精"，往更完美的轨迹去靠拢。
* **当 $\alpha(\mathbf{x}_t) \to 0$ 时（模型滑向了危险、即将碰撞或错误的泥潭区域）**：
公式 (4) 的右半部分趋近于 0，而左半部分 $[1-\alpha] \to 1$。此时吸引力失效（因为周围根本没有好样本），转而全面启动**负向排斥力 $\big[ v^{\text{old}} - v^- \big]$**！模型此时唯一的念头就是："别管去哪了，先逃离眼前这个注定失败的垃圾堆（$v^-$）再说！"

---

## 十四、 完美的数学闭环：策略提升的理论保证

既然证明了 $\Delta$ 可以同时由正、负宇宙的速度差来表达，作者在文中做了一个最强力的**理论收敛性兜底**：

如果我们把这个理想的指导方向 $\Delta$ 带入前文的公式 (3)，并且让指导强度 $\beta$ 恰好等于当前的置信度 $\alpha(\mathbf{x}_t)$：


$$v^*(\mathbf{x}_t, \mathbf{c}, t) = v^{\text{old}}(\mathbf{x}_t, \mathbf{c}, t) + \frac{1}{\alpha(\mathbf{x}_t)} \Delta(\mathbf{x}_t, \mathbf{c}, t)$$

把公式 (4) 的右半部分 $\Delta = \alpha(\mathbf{x}_t) [v^+ - v^{\text{old}}]$ 往里一填：


$$v^* = v^{\text{old}} + \frac{1}{\alpha(\mathbf{x}_t)} \cdot \alpha(\mathbf{x}_t) \big[ v^+ - v^{\text{old}} \big] = v^+(\mathbf{x}_t, \mathbf{c}, t)$$

**系数 $\alpha(\mathbf{x}_t)$ 被完美约掉了！最终的修正目标速度 $v^*$ 毫无偏差地指向了理想专家速度场 $v^+$！**

这就从数学上无条件地保证了：只要我们朝着这个速度场迭代更新当前模型 $v_\theta \to v^*$，更新后的策略就一定满足 **$\pi^* = \pi^+ \succ \pi^{\text{old}}$**，完成绝对的策略提升。

---

## 十五、 总结：这一节突破了什么？

传统的 Diffusion RL 如果想引入负面反馈，往往需要分别训练一个正向模型和一个负向模型，或者写出极度复杂的分布式多目标梯度。

而 Section 3.2 凭借着 Theorem 3.1 惊艳地证明了：**因为正负反馈在扩散速度场上是高度对称、一体两面的，所以我们只需要找一个统一的目标速度场 $v^*$，就能在有监督的框架下，同时完成"遇到好样本吸取经验"和"遇到坏样本紧急避险"的动态双向拉扯。**

这为下一节（也就是我最开始剧透的 Section 3.3 实用 Token 级级联 Loss）提供了最坚实的理论发射台。

这一部分的 **Theorem 3.2 及其优势讨论**是针对扩散模型（Diffusion Models/轨迹规划策略）落地的核心理论实现。

如果说你之前发我的大语言模型（LLM）版本 NFT 是通过 Token 级别的似然比（Likelihood Ratio）在符号序列上做拉扯，那么这一章则是专门针对连续空间（如图像生成、自动驾驶物理轨迹规划）设计的 **DiffusionNFT**。

以下是这一部分的深度拆解：

---

## 十六、 核心定理拆解：Theorem 3.2（隐式参数化与双分支联合优化）

在连续扩散空间中，如果像传统做法那样单独去训练一个正向引导模型和一个负向引导模型，会带来双倍的参数量和极大的采样计算开销。

为了解决这个问题，作者提出了一个极其巧妙的隐式参数化（Implicit Parameterization）训练目标（公式 5）：

$$\mathcal{L}(\theta) = \mathbb{E}_{\mathbf{c}, \pi^{\text{old}}(\mathbf{x}_0|\mathbf{c}), t} \left[ r \|v_\theta^+(\mathbf{x}_t, \mathbf{c}, t) - v\|_2^2 + (1-r)\|v_\theta^-(\mathbf{x}_t, \mathbf{c}, t) - v\|_2^2 \right]$$

### 关键机制：单模型演双角

在这里，我们**只训练一个统一的策略网络 $v_\theta$**。但是，我们在损失函数里通过代数变换，强行让它套上两个不同的外壳，分别去扮演"隐式正策略"和"隐式负策略"：

* **隐式正策略外壳**：$v_\theta^+ := (1-\beta)v^{\text{old}} + \beta v_\theta$
* **隐式负策略外壳**：$v_\theta^- := (1+\beta)v^{\text{old}} - \beta v_\theta$

这两个外壳被塞进了一个由奖励 $r$ 和失败率 $1-r$ 动态调节配比的传统扩散模型有监督 MSE 损失函数中。

### 完美的数学收敛点（公式 6）：

作者通过变分法证明，在数据和网络容量无限的理想状态下，该损失函数的全局最优解 $v_\theta^*$ 恰好满足：


$$v_\theta^*(\mathbf{x}_t, \mathbf{c}, t) = v^{\text{old}}(\mathbf{x}_t, \mathbf{c}, t) + \frac{2}{\beta}\Delta(\mathbf{x}_t, \mathbf{c}, t)$$

**这太漂亮了！** 整个训练过程中，你不需要显式地去计算那个复杂的方向向量 $\Delta$。你只需要让唯一的网络 $v_\theta$ 去跑公式 (5) 的双分支有监督训练，它优化完之后的行为就会天然等价于在原始意志 $v^{\text{old}}$ 的基础上加上了最优的强化指导方向 $\Delta$。

---

## 十七、 DiffusionNFT 的四大终极优势

在这篇截图的下半部分，作者列举了这种有监督对齐范式相比于传统基于策略梯度（Policy Gradient）的扩散强化学习（如 FlowGRPO、DDPO）的压倒性优势：

### 17.1 前向一致性（Forward Consistency）

* **传统痛点**：像 FlowGRPO 这样的方法直接在逆向去噪过程（Reverse Process）上定义策略梯度。这会导致模型在迭代更新后，原本前向加噪过程的边际概率密度与逆向去噪完全脱节，破坏了潜在的物理约束（违背了 Fokker-Planck 福克-普朗克方程）。
* **NFT 解法**：DiffusionNFT 将 Loss 直接定义在前向加噪过程（Forward Process）上（经典的典型扩散损失）。这确保了无论策略怎么变，其底层概率密度始终严格遵循合法的随机微分方程映射，保证了生成的轨迹或图像符合底层物理分布。

### 17.2 求解器灵活性（Solver Flexibility）

* **传统痛点**：传统的扩散 RL 训练和数据采样紧密耦合，要求必须完整记录并存储采样过程中的**整条去噪轨迹链条**（所有中间时步的 $t$ 状态），不仅极度消耗显存，还限制了你只能使用特定的一阶 SDE 采样器。
* **NFT 解法**：它把策略微调和数据采样**彻底解耦**。在线采样时，你可以用任何现有的高效黑盒加速求解器（如 DPM-Solver++、Flow Matching DDIM）；而训练时，你**只需要收集最终生成的干净样本 $\mathbf{x}_0$ 及其对应的奖励 $r$**。中间的中间状态 $\mathbf{x}_t$ 是在训练时直接根据加噪公式在线随机生成的。显存开销直接断崖式下跌。

### 17.3 隐式指导集成（Implicit Guidance Integration）

* **传统痛点**：传统的分类器引导（Classifier Guidance）需要在线去噪时额外挂载一个打分网络来实时算梯度，拖慢推理速度。
* **NFT 解法**：如前文所述，强化指导力 $\Delta$ 被隐式地熔铸（Parameterize）进了当前这一个策略网络 $v_\theta$ 中。在推理部署时，你不需要做任何的外加引导修改，直接像正常无条件扩散模型一样一步去噪即可，模型自己就知道怎么避障或输出高分样本。

### 17.4 免似然估计表征（Likelihood-Free Formulation）

* **传统痛点**：由于扩散模型本质上很难精确计算精确的似然概率密度 $\log \pi(\mathbf{x})$，前人的策略梯度微调（如基于 PPO 的框架）不得不引入极其复杂的变分下界（SDE 离散化近似、Jensen 不等式放大等）。这些近似会给训练梯度带来不可控的**系统性估计偏差（Estimation Bias）**。
* **NFT 解法**：纯粹的监督学习（SL）MSE 损失，**天然不需要计算任何似然概率**。从根源上绕过了这个数学陷阱，实现了绝对的数值鲁棒性。

---

## 总结：两条落地支线的对比

读到这里，整篇论文的宏大构架已经完全清晰了。作者用同一个理论（控制即推断），在两个完全不同的工程领域各自开花：

1. **在大语言模型（LLM）等离散符号序列上**：演化为了你在上一轮发给我的 Section 3.3，通过 Token 级似然比截断和 STE 门控来实现稳定训练。
2. **在连续空间（自动驾驶轨迹规划、物理世界扩散生成）上**：演化为了现在的 Section 3.2，通过加噪前向过程的单模型隐式双分支 MSE 联合优化来实现高效落地。


**扩散模型（Diffusion Policy/轨迹规划）** 领域的**全套工程落地指南（Section 3.3 实用算法与工程设计）**

---

## 十八、 算法核心流程：Algorithm 1 伪代码拆解

整个在线强化学习的迭代被极其优雅地拆成了两大步：

### 18.1 采样阶段（Rollout Step，第 2-7 行）

* 对于给定的 Prompt（或自动驾驶场景 $c$），让当前的采样老模型 $v^{\text{old}}$ 去连续生成 $K$ 个轨迹/图像样本 $\mathbf{x}_0^{1:K}$。
* 环境或奖励打分模型（如碰撞检测器、舒适度指标）给出这些样本的原始连续分数 $r^{\text{raw}}$。
* **组内归一化（第 4-5 行）**：算出这 $K$ 个样本的均值，进行类似于 GRPO 的组内优势归一化，然后通过一个带截断的映射，强行把得分映射到 $[0, 1]$ 区间，作为"最优性概率 $r$"。分高的 $r \to 1$（好学生），分低的 $r \to 0$（坏教材）。

### 18.2 梯度更新阶段（Gradient Step，第 8-13 行）

* **加噪（第 9 行）**：直接根据前向加噪公式，对干净样本 $\mathbf{x}_0$ 随机撒上噪声 $\epsilon$，得到中间状态 $\mathbf{x}_t$ 和目标真实速度 $v$。
* **套壳（第 10-11 行）**：用你当前正在训练的唯一网络 $v_\theta$，结合老策略 $v^{\text{old}}$，隐式组装出正、负两个虚拟速度 $v_\theta^+$ 和 $v_\theta^-$。
* **拉扯（第 12 行）**：直接计算有监督的 MSE 损失更新参数 $\theta$：

$$\mathcal{L} = r \| v_\theta^+ - v \|_2^2 + (1-r) \| v_\theta^- - v \|_2^2$$



---

## 十九、 四大核心落地工程设计（Key Design Choices）

为了在工业级分布式训练中保证这个算法不崩溃、收敛快，作者提出了四个关键补丁：

### 19.1 Optimality Reward（连续奖励的"软二进制化"）

* **痛点**：Levine 的"控制即推断"理论要求奖励必须是个概率 $r \in [0, 1]$。但现实中（尤其是自动驾驶或画图），打分模型给出的奖励都是无约束的连续实数（比如 $-100$ 到 $+10$ 的碰撞/距离惩罚）。
* **解法**：借用大模型 **GRPO** 的精髓，对同一个 Prompt 生成的 $K$ 个回答进行**组内减均值**（用组内平均分作为 Baseline 估计），再除以标准差 $Z_c$，最后用 `clip` 强行锁死在 $\pm 1$ 之间，缩放到 $[0, 1]$ 内。这不仅满足了数学假设，还自动实现了"水涨船高"的相对优势对比。

### 19.2 Soft Update of Sampling Policy（采样策略的 EMA 软更新）

* **痛点**：如果是绝对的 On-policy（即一更新完 $\theta$ 就立刻让老策略 $v^{\text{old}} \leftarrow v_\theta$），训练会由于新旧策略偏差过大而变得极度不稳定，极易发生毁灭性崩溃（Catastrophic Collapse）；但如果是纯 Offline（不更新采样策略），模型又会收敛极慢。
* **解法**：采用 **EMA（指数移动平均）软更新**：

$$\theta^{\text{old}} \leftarrow \eta_i \theta^{\text{old}} + (1 - \eta_i) \theta$$



通过调节参数 $\eta$ 完美平衡了在线学习的速度与离线学习的稳定性。

### 19.3 Adaptive Loss Weighting（免调参的自适应时步加权）

* **痛点**：传统的 Diffusion 训练需要针对不同的去噪时步 $t$ 手动设计权重函数 $w(t)$。调这个超参数如同玄学，极其耗费精力。
* **解法**：作者直接将预测速度 $v_\theta$ 转换成对干净样本 $\mathbf{x}_0$ 的预测器 $\mathbf{x}_theta$。然后引入大名鼎鼎的 **DMD（Diffusion Distillation）** 机制，利用停止梯度算子（Stop-Gradient, `sg`）对误差进行了自适应的归一化处理：

$$w(t)\|v_\theta - v\|_2^2 \leftarrow \frac{\|\mathbf{x}_\theta - \mathbf{x}_0\|_2^2}{\text{sg}(\text{mean}(\text{abs}(\mathbf{x}_\theta - \mathbf{x}_0)))}$$



在不需要人肉调参的前提下，直接实现了训练速度的暴增。

### 19.4 CFG-Free Optimization（免 Classifier-Free 引导的轻量化推理）

* **痛点**：文生图或高精度规划通常默认挂载 CFG（分类器免引导），在推理时强行算两遍网络（条件和无条件）来放大控制力，这直接让推理耗时翻倍，效率极其低下。
* **解法**：作者从理论上解构了 CFG——**CFG 本质上就是一种手写的、离线形式的强化引导（正条件充当吸引力，无条件充当负排斥力）**。
既然 DiffusionNFT 的在线 RL 微调本身就在做正负拉扯，那我们在初始化时**直接把无条件的分支扔掉，只用条件模型初始化**。实验表明，虽然刚开始性能一般，但经过 NFT 强化对齐后，单网络模型的性能迅速飙升并彻底超越了双倍开销的 CFG Baseline！这意味着**模型在强化训练中自己隐式学会了 CFG 的提纯能力，推理时却只需要一半的算力**。

---

## 二十、 跨文本与连续空间的统一图像

配合你发的第一张图来看，你会发现作者的野心很大：

* 在 **LLM 文本端**：遇到了垃圾 Token，使用 Unlikelihood 显式地惩罚 $\log(1-p_{\theta})$。
* 在 **Diffusion 连续端**：遇到了低分样本（$r \to 0$），通过公式 (5) 的 $(1-r) \| v_{\theta}^- - v \|_2^2$ 分支，让 $v_\theta^-$ 逼近真噪，从而隐式地驱动当前策略 $v_\theta$ 向着相反的方向（即远离垃圾样本的速度）疯狂逃跑。
